# 02 — Logic, Waveforms, MMIO, and UART Traces

**Modules:** `00.08`, `01.02`, `02.02`  
**Prerequisite:** notebook 01 and its explicit 16-bit little-endian byte contract.  
**Consumes:** `fromthetransistor/01-byte-contract.json`  
**Emits:** `fromthetransistor/02-uart-trace.json`

We now make stored bytes observable as clocked serial logic. The model is intentionally smaller than a physical UART: one start bit, eight LSB-first data bits, one stop bit, and a memory-mapped transmit register. Its value is that every cycle and boundary is inspectable before hardware synthesis.

In [ ]:
from pathlib import Path
import json
import os

ARTIFACT_ROOT = Path(os.environ['COURSE_NOTEBOOK_ARTIFACTS'])
contract_path = ARTIFACT_ROOT / 'fromthetransistor/01-byte-contract.json'
byte_contract = json.loads(contract_path.read_text(encoding='utf-8'))
assert byte_contract['schema_version'] == 1
assert byte_contract['word_bits'] == 16
assert byte_contract['byteorder'] == 'little'
upstream_payload = bytes.fromhex(byte_contract['known_word_hex'])
assert upstream_payload == bytes([0xD6, 0xFF])


## Worked analysis

A hardware trace is a time-indexed claim. Sampling in the middle of each bit should reconstruct the frame, while a bad start or stop boundary must fail loudly. The MMIO model separates a CPU-visible address contract from the serial state machine. A counter-derived LED is included as the smallest stateful logic, but a simulated toggle is not proof of correct FPGA pins, voltage, clocks, or timing closure.

In [ ]:
def half_adder(a, b):
    if a not in (0, 1) or b not in (0, 1):
        raise ValueError('logic values must be 0 or 1')
    return a ^ b, a & b

def uart_frame(value):
    if not 0 <= value <= 0xFF:
        raise ValueError('UART payload must be one byte')
    return [0] + [(value >> bit) & 1 for bit in range(8)] + [1]

def decode_uart(frame):
    if len(frame) != 10 or frame[0] != 0 or frame[-1] != 1:
        raise ValueError('invalid 8N1 frame boundary')
    return sum(frame[1 + bit] << bit for bit in range(8))

def oversample(frame, samples_per_bit=4):
    if samples_per_bit < 2:
        raise ValueError('need at least two samples per bit')
    return [level for level in frame for _ in range(samples_per_bit)]

class UartMMIO:
    TX = 0xF000
    def __init__(self):
        self.frames = []
    def write8(self, address, value):
        if address != self.TX:
            raise KeyError(f'unmapped MMIO address 0x{address:04x}')
        self.frames.append(uart_frame(value))

assert [half_adder(a, b) for a in (0, 1) for b in (0, 1)] == [(0, 0), (1, 0), (1, 0), (0, 1)]
device = UartMMIO()
for byte in upstream_payload:
    device.write8(UartMMIO.TX, byte)
decoded = bytes(decode_uart(frame) for frame in device.frames)
waveforms = [oversample(frame) for frame in device.frames]
assert decoded == upstream_payload
assert all(len(waveform) == 40 for waveform in waveforms)
led_trace = [{'cycle': cycle, 'led': (cycle // 4) & 1} for cycle in range(16)]
assert [sample['led'] for sample in led_trace] == [0] * 4 + [1] * 4 + [0] * 4 + [1] * 4

artifact = {
    'schema_version': 1,
    'format': '8N1-lsb-first',
    'mmio_tx': UartMMIO.TX,
    'payload_hex': upstream_payload.hex(),
    'frames': device.frames,
    'samples_per_bit': 4,
    'waveforms': waveforms,
    'led_trace': led_trace,
}
target = ARTIFACT_ROOT / 'fromthetransistor/02-uart-trace.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(artifact, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('decoded UART payload:', decoded.hex())


## Exercise and boundary checks

1. Predict and draw the 8N1 frame for `0x41` before encoding it.
2. Flip its stop bit and require a framing error.
3. Try an adjacent unmapped MMIO address and require an address fault.
4. Add a `busy` flag and decide whether a write stalls, queues, or fails. State the contract before coding it.

In [ ]:
exercise_frame = uart_frame(0x41)
assert exercise_frame == [0, 1, 0, 0, 0, 0, 0, 1, 0, 1]
broken = exercise_frame.copy()
broken[-1] = 0
try:
    decode_uart(broken)
    raise AssertionError('bad stop bit was accepted')
except ValueError:
    pass
try:
    device.write8(UartMMIO.TX + 1, 0)
    raise AssertionError('unmapped MMIO write was accepted')
except KeyError:
    pass
assert decode_uart(exercise_frame) == 0x41
print('UART boundary self-checks passed')
